# plate-redactor — Phase 2: train YOLOv8n

Self-contained notebook to train the single-class licence-plate detector on
Kaggle (P100) or Colab (T4). It installs deps, gets the synthetic dataset,
trains YOLOv8n, and saves `best.pt` somewhere persistent.

**Architecture:** YOLOv8n (nano) — small footprint, native TFLite export with
NMS baked in, Apache-2.0. See `src/training/README.md` for the full rationale.

> **Checkpoints are never committed to Git.** They go to `/kaggle/working/` or
> Google Drive. The repo's `.gitignore` already excludes `models/`, `*.pt`, etc.

## Rotating between Kaggle and Colab (§4 of the work order)

Free GPU time is rate-limited, so rotate platforms:

| Platform | GPU | Quota | Persistence |
|---|---|---|---|
| **Kaggle** | P100 | 30 GPU-hrs/week | `/kaggle/working/` output (download the run) |
| **Colab Free** | T4 | ~15–30 hrs/week, 12-hr session cap, no guarantee | Mount Google Drive |

**Strategy**

- Start on **Kaggle** (more reliable GPU, weekly budget). Add the synthetic
  dataset as a *Kaggle Dataset* input and read it from `/kaggle/input/...`.
- If Kaggle's weekly quota is exhausted, switch to **Colab**: mount Drive,
  regenerate or upload the dataset, and train there.
- `save_period=10` (set in `train.py`) checkpoints every 10 epochs so a dropped
  12-hr Colab session loses at most 10 epochs — resume from the last checkpoint.
- **Record which platform produced the released checkpoint** in `WORKING_STATE.md`.

The cells below auto-detect the platform and set paths accordingly.

## 1. Install dependencies

In [ ]:
# Ultralytics pulls a compatible torch. On Kaggle/Colab a CUDA torch is usually
# preinstalled, so this is fast.
%pip install -q ultralytics

import ultralytics, torch
ultralytics.checks()
print('CUDA available:', torch.cuda.is_available())

## 2. Get the code + detect platform

In [ ]:
import os, sys, subprocess
from pathlib import Path

ON_KAGGLE = Path('/kaggle').exists()
# Kaggle wins if both look true; only treat as Colab when not on Kaggle.
ON_COLAB = (not ON_KAGGLE) and ('google.colab' in sys.modules or Path('/content').exists())
print('Kaggle:', ON_KAGGLE, '| Colab:', ON_COLAB)

# Clone the repo to get the generator + training scripts.
REPO_URL = 'https://github.com/Andre-Ehret/plate-redactor.git'
WORK = Path('/kaggle/working') if ON_KAGGLE else Path('/content')
REPO = WORK / 'plate-redactor'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
%pip install -q -e {str(REPO)}
os.chdir(REPO)
print('cwd:', os.getcwd())

## 3. Get the synthetic dataset

Two options — pick one:

- **A. Regenerate inline** (no real photos needed; uses synthetic fallback
  backgrounds, or real backgrounds if you attach them). Fully reproducible.
- **B. Attach a prebuilt Kaggle Dataset** with real-photo backgrounds composited
  in, and point `DATA_YAML` at it.

For a better detector, supply real vehicle/street backgrounds via
`--backgrounds` (see the root README). Below uses option A.

In [ ]:
# Option A — regenerate inline. Bump --n for a real run (e.g. 5000).
# Attach backgrounds and add: --backgrounds /kaggle/input/<your-bg-dataset>
subprocess.run([
    sys.executable, '-m', 'plate_redactor.generator.generate',
    '--n', '5000', '--seed', '42', '--out', 'data/synthetic',
], check=True)

DATA_YAML = REPO / 'data' / 'synthetic' / 'data.yaml'
print('data.yaml:', DATA_YAML, '— exists:', DATA_YAML.exists())

# Option B — instead point at an attached Kaggle Dataset, e.g.:
# DATA_YAML = Path('/kaggle/input/plate-synthetic/data.yaml')

## 4. (Colab only) Mount Drive for checkpoint persistence

In [ ]:
DRIVE_OUT = None
if ON_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_OUT = Path('/content/drive/MyDrive/plate-redactor')
        DRIVE_OUT.mkdir(parents=True, exist_ok=True)
        print('Checkpoints will be copied to:', DRIVE_OUT)
    except Exception as e:
        # Some Colab runtimes (local/Enterprise/TPU) don't support drive.mount.
        # Not fatal — training is unaffected; best.pt is downloaded directly later.
        print(f'Drive mount unavailable ({type(e).__name__}) — skipping. '
              'best.pt will be offered as a direct download in step 7.')
        DRIVE_OUT = None
else:
    print('Not on Colab — skipping Drive mount.')

## 5. Train

Calls `src/training/train.py` with the documented defaults (imgsz=320,
epochs=100, batch=16). It copies the best checkpoint to `models/best.pt` and
prints the final recall/precision/mAP.

In [ ]:
subprocess.run([
    sys.executable, 'src/training/train.py',
    '--data', str(DATA_YAML),
    '--epochs', '100',
    '--imgsz', '320',
    '--batch', '16',
    '--seed', '0',
], check=True)

## 6. Inspect loss curves + metrics

In [ ]:
from IPython.display import Image as IPyImage, display
run_dir = sorted((REPO / 'models' / 'runs').glob('plate_yolov8n*'))[-1]
print('Run dir:', run_dir)
for img in ['results.png', 'PR_curve.png', 'confusion_matrix.png']:
    p = run_dir / img
    if p.exists():
        print(img)
        display(IPyImage(filename=str(p)))

## 7. Save / download `best.pt`

**Never commit this to Git.** Persist it as a run output (Kaggle) or to Drive
(Colab), then attach it to a GitHub Release for the app to pin.

In [ ]:
import shutil
best = REPO / 'models' / 'best.pt'
assert best.exists(), 'best.pt missing — did training finish?'

if ON_KAGGLE:
    shutil.copy2(best, Path('/kaggle/working/best.pt'))
    print('Saved to /kaggle/working/best.pt — download it from the Kaggle run output.')
else:
    # Colab (or anything else): copy to Drive if mounted, then offer a browser download.
    if DRIVE_OUT is not None:
        shutil.copy2(best, DRIVE_OUT / 'best.pt')
        print('Saved to', DRIVE_OUT / 'best.pt')
    if ON_COLAB:
        try:
            from google.colab import files
            files.download(str(best))  # trigger a browser download
        except Exception as e:
            print(f'Auto-download unavailable ({type(e).__name__}). best.pt at:', best)
    else:
        print('best.pt at:', best)

## 8. (Optional) Sanity check on CPU

In [ ]:
subprocess.run([sys.executable, 'src/training/sanity_check.py', '--n', '5'], check=True)